# 03. Cascade 평가와 credential event

목표: fast-path/vision cascade의 품질·latency·비용을 함께 평가하고 원본 screenshot 없이 검증 가능한 completion event를 만드는 방법을 실습합니다.

In [ ]:
from collections import Counter
import hashlib
import hmac
import json
import random

rng = random.Random(42)
records = []
for _ in range(1000):
    truth = rng.random() < 0.60
    fast_path = rng.random() < 0.72
    if fast_path:
        prediction = truth if rng.random() < 0.995 else not truth
        latency_ms, cost_units, route = 50, 0.0, "dom"
    else:
        prediction = truth if rng.random() < 0.93 else not truth
        latency_ms, cost_units, route = 2000, 1.0, "vision"
    records.append((truth, prediction, latency_ms, cost_units, route))

counts = Counter((truth, prediction) for truth, prediction, *_ in records)
false_accept = counts[(False, True)] / max(1, sum(1 for r in records if not r[0]))
false_reject = counts[(True, False)] / max(1, sum(1 for r in records if r[0]))
summary = {
    "fast_path_rate": sum(r[4] == "dom" for r in records) / len(records),
    "false_accept_rate": false_accept,
    "false_reject_rate": false_reject,
    "mean_latency_ms": sum(r[2] for r in records) / len(records),
    "mean_cost_units": sum(r[3] for r in records) / len(records),
}
summary

위 수치는 원문의 benchmark를 재현한 것이 아니라 trade-off를 보기 위한 simulation입니다. credentialing에서는 false accept를 일반 안내의 false reject보다 더 비싸게 취급할 수 있습니다. threshold는 위험 수준별로 달라야 합니다.

In [ ]:
def create_completion_event(user_id: str, skill_id: str, evidence: dict, secret: bytes) -> dict:
    # 원본 screenshot 대신 정규화된 최소 evidence의 digest를 기록합니다.
    canonical = json.dumps(evidence, sort_keys=True, separators=(",", ":"))
    evidence_digest = hashlib.sha256(canonical.encode()).hexdigest()
    event = {
        "schema_version": "1.0",
        "subject": user_id,
        "skill": skill_id,
        "evidence_digest": evidence_digest,
        "evaluator": "cascade-policy-v3",
        "status": "completed",
    }
    body = json.dumps(event, sort_keys=True, separators=(",", ":")).encode()
    event["signature"] = hmac.new(secret, body, hashlib.sha256).hexdigest()
    return event

secret = b"demo-only-secret-use-a-secret-manager-in-production"
event = create_completion_event("employee-42", "crm-search-v2", {"backend_event_id": "evt-123"}, secret)
print(json.dumps(event, indent=2))

## 보안 주의

HMAC 예제는 한 조직 안에서 위변조를 탐지하는 toy mechanism입니다. 공개 검증 credential에는 표준 서명, key rotation, issuer identity, expiry와 revocation이 필요합니다. raw screenshot 삭제 여부, evidence 접근 권한과 이의 제기 절차도 함께 설계하세요.